In [17]:
import cv2
import mediapipe as mp
import numpy as np

def calculate_distance(known_width, focal_length, perceived_width):
    """Estimate distance based on the width of the detected hand."""
    if perceived_width == 0:
        return None
    return (known_width * focal_length) / perceived_width

# Initialize MediaPipe Hand Detection
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

# Open Camera
cap = cv2.VideoCapture(0)

# Reference measurements (calibration required)
KNOWN_WIDTH = 8.0  # Average width of a human hand in cm
FOCAL_LENGTH = 500  # Approximate focal length (needs calibration)

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Flip image for a mirror effect
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        
        # Convert to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)
        
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # Get bounding box around the hand
                x_min = w
                y_min = h
                x_max = 0
                y_max = 0
                
                for landmark in hand_landmarks.landmark:
                    x, y = int(landmark.x * w), int(landmark.y * h)
                    x_min = min(x, x_min)
                    y_min = min(y, y_min)
                    x_max = max(x, x_max)
                    y_max = max(y, y_max)
                
                hand_width_pixels = x_max - x_min
                estimated_distance = calculate_distance(KNOWN_WIDTH, FOCAL_LENGTH, hand_width_pixels)
                
                # Draw bounding box
                cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
                
                if estimated_distance:
                    cv2.putText(frame, f"Distance: {estimated_distance:.2f} cm", (x_min, y_min - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
                
                # Draw hand landmarks
                mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        
        # Show frame
        cv2.imshow("Hand Distance Measurement", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\n[INFO] 'q' pressed, exiting gracefully...")
            break

except KeyboardInterrupt:
    print("\n[INFO] Keyboard Interrupt detected, exiting...")

finally:
    print("[INFO] Releasing resources...")
    cap.release()
    cv2.destroyAllWindows()


[INFO] 'q' pressed, exiting gracefully...
[INFO] Releasing resources...
